# Solar Filament Segmentation - Version 25 uploaded ZIP validated

Offline inference for the already attached `best_solar_unet` Kaggle Dataset. This notebook detects Kaggle's extracted PyTorch archive, reconstructs it when needed, maps the EfficientNet-B3 encoder weights to torchvision, recreates the SMP 0.3.3 Unet++ decoder, and validates the official compressed COCO RLE submission.

In [ ]:
import glob, os, zipfile
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b3
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SIZE = (512, 512)
OUTPUT_SIZE = (2048, 2048)
THRESHOLD = 0.50
BATCH_SIZE = 2
NUM_WORKERS = 2
OUTPUT_PATH = '/kaggle/working/submission_v25_uploaded_zip_validated.csv'

def find_test_dir():
    patterns = [
        '/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026/test/test_images',
        '/kaggle/input/**/test/test_images',
    ]
    for pattern in patterns:
        found = sorted(glob.glob(pattern, recursive=True))
        if found:
            return found[0]
    raise FileNotFoundError('No test/test_images directory found under /kaggle/input.')

def repack_extracted_checkpoint(root):
    archive = '/kaggle/working/repacked_best_solar_unet.zip'
    root_path = Path(root)
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as handle:
        for path in root_path.rglob('*'):
            if path.is_file():
                handle.write(path, path.relative_to(root_path.parent).as_posix())
    return archive

def find_checkpoint():
    direct_patterns = [
        '/kaggle/input/**/best_solar_unet.pth',
        '/kaggle/input/**/best_solar_unet.pt',
        '/kaggle/input/**/best_solar_unet.zip',
    ]
    for pattern in direct_patterns:
        found = sorted(glob.glob(pattern, recursive=True))
        if found:
            return found[0]
    roots = []
    for marker in glob.glob('/kaggle/input/**/data.pkl', recursive=True):
        root = os.path.dirname(marker)
        if os.path.isdir(os.path.join(root, 'data')) and os.path.isfile(os.path.join(root, 'byteorder')):
            roots.append(root)
    if roots:
        return repack_extracted_checkpoint(sorted(set(roots))[0])
    visible = sorted(glob.glob('/kaggle/input/**', recursive=True))[:50]
    raise FileNotFoundError(f'No PyTorch checkpoint found. Visible paths: {visible}')

TEST_IMG_DIR = find_test_dir()
CHECKPOINT_PATH = find_checkpoint()
print({'device': str(DEVICE), 'test_dir': TEST_IMG_DIR, 'checkpoint': CHECKPOINT_PATH})

In [ ]:
class Conv2dReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv1 = Conv2dReLU(in_channels + skip_channels, out_channels)
        self.conv2 = Conv2dReLU(out_channels, out_channels)
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv2(self.conv1(x))

class UnetPlusPlusDecoder(nn.Module):
    def __init__(self, encoder_channels=(3,40,32,48,136,384), decoder_channels=(256,128,64,32,16)):
        super().__init__()
        channels = list(encoder_channels[1:])[::-1]
        self.in_channels = [channels[0]] + list(decoder_channels[:-1])
        self.skip_channels = list(channels[1:]) + [0]
        self.out_channels = list(decoder_channels)
        blocks = {}
        for layer_idx in range(len(self.in_channels) - 1):
            for depth_idx in range(layer_idx + 1):
                if depth_idx == 0:
                    in_ch = self.in_channels[layer_idx]
                    skip_ch = self.skip_channels[layer_idx] * (layer_idx + 1)
                    out_ch = self.out_channels[layer_idx]
                else:
                    out_ch = self.skip_channels[layer_idx]
                    skip_ch = self.skip_channels[layer_idx] * (layer_idx + 1 - depth_idx)
                    in_ch = self.skip_channels[layer_idx - 1]
                blocks[f'x_{depth_idx}_{layer_idx}'] = DecoderBlock(in_ch, skip_ch, out_ch)
        blocks[f'x_0_{len(self.in_channels)-1}'] = DecoderBlock(self.in_channels[-1], 0, self.out_channels[-1])
        self.blocks = nn.ModuleDict(blocks)
        self.depth = len(self.in_channels) - 1
    def forward(self, *features):
        features = features[1:][::-1]
        dense_x = {}
        for layer_idx in range(len(self.in_channels) - 1):
            for depth_idx in range(self.depth - layer_idx):
                if layer_idx == 0:
                    output = self.blocks[f'x_{depth_idx}_{depth_idx}'](features[depth_idx], features[depth_idx + 1])
                    dense_x[f'x_{depth_idx}_{depth_idx}'] = output
                else:
                    dense_l_i = depth_idx + layer_idx
                    cat_features = [dense_x[f'x_{idx}_{dense_l_i}'] for idx in range(depth_idx + 1, dense_l_i + 1)]
                    cat_features = torch.cat(cat_features + [features[dense_l_i + 1]], dim=1)
                    dense_x[f'x_{depth_idx}_{dense_l_i}'] = self.blocks[f'x_{depth_idx}_{dense_l_i}'](dense_x[f'x_{depth_idx}_{dense_l_i-1}'], cat_features)
        dense_x[f'x_0_{self.depth}'] = self.blocks[f'x_0_{self.depth}'](dense_x[f'x_0_{self.depth-1}'])
        return dense_x[f'x_0_{self.depth}']

class OfflineSMPUnetPlusPlus(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = efficientnet_b3(weights=None)
        self.features = backbone.features
        self.decoder = UnetPlusPlusDecoder()
        self.segmentation_head = nn.Sequential(nn.Conv2d(16, 1, 3, padding=1))
    def encode(self, x):
        result = [x]
        x = self.features[0](x)
        result.append(x)
        block_count = 0
        stage_ends = {5, 8, 18, 26}
        for stage in self.features[1:8]:
            for block in stage:
                x = block(x)
                block_count += 1
                if block_count in stage_ends:
                    result.append(x)
        if block_count != 26 or len(result) != 6:
            raise RuntimeError(f'Unexpected EfficientNet-B3 layout: blocks={block_count}, features={len(result)}')
        return result
    def forward(self, x):
        return self.segmentation_head(self.decoder(*self.encode(x)))

def load_offline_model(checkpoint_path):
    state = torch.load(checkpoint_path, map_location='cpu')
    if 'state_dict' in state:
        state = state['state_dict']
    model = OfflineSMPUnetPlusPlus()
    source_encoder = [(k, v) for k, v in state.items() if k.startswith('encoder.') and not k.startswith(('encoder._conv_head', 'encoder._bn1'))]
    target_encoder = [(k, v) for k, v in model.features.state_dict().items() if not k.startswith('8.')]
    if len(source_encoder) != len(target_encoder):
        raise RuntimeError(f'Encoder tensor count mismatch: {len(source_encoder)} vs {len(target_encoder)}')
    mapped = {}
    for (source_key, source_value), (target_key, target_value) in zip(source_encoder, target_encoder):
        if source_value.shape != target_value.shape:
            raise RuntimeError(f'Encoder shape mismatch: {source_key} -> {target_key}')
        mapped[target_key] = source_value
    missing, unexpected = model.features.load_state_dict(mapped, strict=False)
    allowed_missing = {k for k in missing if k.startswith('8.')}
    if set(missing) != allowed_missing or unexpected:
        raise RuntimeError(f'Encoder load mismatch: missing={missing[:3]}, unexpected={unexpected[:3]}')
    decoder_state = {k.removeprefix('decoder.'): v for k, v in state.items() if k.startswith('decoder.')}
    head_state = {k.removeprefix('segmentation_head.'): v for k, v in state.items() if k.startswith('segmentation_head.')}
    model.decoder.load_state_dict(decoder_state, strict=True)
    model.segmentation_head.load_state_dict(head_state, strict=True)
    return model.to(DEVICE).eval()

model = load_offline_model(CHECKPOINT_PATH)
print('Checkpoint loaded into offline EfficientNet-B3 Unet++.')

In [ ]:
class TestDataset(Dataset):
    def __init__(self, directory):
        extensions = ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG')
        self.paths = sorted(path for ext in extensions for path in glob.glob(os.path.join(directory, ext)))
        if not self.paths:
            raise FileNotFoundError(f'No test images found in {directory}')
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, index):
        path = self.paths[index]
        image = cv2.imread(path, cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f'Cannot read {path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, INPUT_SIZE, interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
        image = (image - np.array([.485,.456,.406], np.float32)) / np.array([.229,.224,.225], np.float32)
        return torch.from_numpy(image.transpose(2,0,1)).float(), os.path.basename(path)

@torch.inference_mode()
def predict(loader):
    names, masks = [], []
    for images, batch_names in tqdm(loader, desc='Inference'):
        probabilities = torch.sigmoid(model(images.to(DEVICE, non_blocking=True)))[:,0].cpu().numpy()
        for name, probability in zip(batch_names, probabilities):
            probability = cv2.resize(probability, OUTPUT_SIZE, interpolation=cv2.INTER_LINEAR)
            names.append(name)
            masks.append((probability >= THRESHOLD).astype(np.uint8))
    return names, masks

loader = DataLoader(TestDataset(TEST_IMG_DIR), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
image_names, predicted_masks = predict(loader)
print(f'Predicted {len(predicted_masks)} image masks.')

In [ ]:
def coco_compress_counts(counts):
    chars = []
    for index, count in enumerate(counts):
        value = int(count)
        if index > 2:
            value -= int(counts[index - 2])
        more = True
        while more:
            char = value & 0x1f
            value >>= 5
            more = value != -1 if (char & 0x10) else value != 0
            if more:
                char |= 0x20
            chars.append(chr(char + 48))
    return ''.join(chars)

def encode_coco_rle(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    if mask.shape != OUTPUT_SIZE:
        raise ValueError(f'Expected {OUTPUT_SIZE}, got {mask.shape}')
    if not np.any(mask):
        return None
    pixels = np.asfortranarray(mask).reshape(-1, order='F')
    changes = np.flatnonzero(np.diff(np.concatenate(([0], pixels, [0]))))
    counts, previous = [], 0
    for start, end in zip(changes[0::2], changes[1::2]):
        counts.extend([int(start - previous), int(end - start)])
        previous = int(end)
    if previous < pixels.size:
        counts.append(int(pixels.size - previous))
    return coco_compress_counts(counts)

rows = []
for name, mask in zip(image_names, predicted_masks):
    rle = encode_coco_rle(mask)
    if rle is not None and rle.strip():
        rows.append({'filament_id': f'{Path(name).stem}_1', 'segmentation_rle': rle})
submission = pd.DataFrame(rows, columns=['filament_id','segmentation_rle'])
if submission.empty or submission.isna().any().any():
    raise ValueError('Submission is empty or contains null values.')
if submission['segmentation_rle'].astype(str).str.strip().eq('').any():
    raise ValueError('Submission contains an empty RLE.')
if submission['filament_id'].duplicated().any():
    raise ValueError('Submission contains duplicate IDs.')
submission.to_csv(OUTPUT_PATH, index=False, lineterminator='\n')
checked = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
assert checked.columns.tolist() == ['filament_id','segmentation_rle']
assert not checked.empty and not checked.isna().any().any()
assert not checked['segmentation_rle'].astype(str).str.strip().eq('').any()
print(f'VALIDATED: {OUTPUT_PATH} ({len(checked)} rows)')
print(checked.head())